In [1]:
import OrcFxAPI
from pathlib import Path
import numpy as np

owd_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_data\Merganser_final_compensated.owd")
owr_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_results\test_40ton.owr")
xlsx_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\XLSX_results\test_40ton.xlsx")


In [2]:
water_depth = 30.0  # m
wave_periods_coarse = [1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9]
  # jouw coarse

wave_periods_fine = np.arange(1, 8.01, 1)

wave_periods_rest = [8.1, 8.2, 8.3, 8.4, 8.5, 8.6, 8.7, 8.8, 8.9, 9.0, 9.5, 10.0, 10.5, 11, 11.5, 12, 13.5, 14, 14.5, 15, 15.5, 16, 17, 18, 19, 20 ]  # jouw rest

wave_periods = np.concatenate([
    # wave_periods_coarse,
    wave_periods_fine,
    wave_periods_rest
])
wave_headings = [0.0, 45.0, 90.0, 135.0, 180.0]
   # deg

mass = 40  # ton, alleen goed als jouw model ook ton verwacht
com_x = 0.0
com_y = 0.0
com_z = 8.18
# voorbeeld inertia matrix rond CoM
Ixx = 3289 
Iyy = 3334
Izz = 5409
Ixy = 0.0
Ixz = 0.0
Iyz = 0.0


# =========================
# HELPER FUNCTIONS
# =========================
def set_value(obj, name, value):
    """
    Zet een OrcaWave data item.
    De exacte data-itemnamen moet je uit OrcaWave halen met F7.
    """
    try:
        obj[name] = value
        print(f"Set: {name} = {value}")
    except Exception as e:
        print(f"Kon data item niet zetten: {name}")
        print(f"  Waarde: {value}")
        print(f"  Fout: {e}")


def print_validation(diff):
    info = getattr(diff, "ValidationInformationText", "")
    warnings = getattr(diff, "ValidationWarningText", "")
    errors = getattr(diff, "ValidationErrorText", "")

    print("\n=== VALIDATION INFO ===")
    print(info if info else "(geen info)")

    print("\n=== VALIDATION WARNINGS ===")
    print(warnings if warnings else "(geen warnings)")

    print("\n=== VALIDATION ERRORS ===")
    print(errors if errors else "(geen errors)")

    return errors


# =========================
# MAIN
# =========================
diff = OrcFxAPI.Diffraction()
diff.LoadData(str(owd_file))

# -------------------------------------------------
# BELANGRIJK:
# De namen hieronder zijn PLACEHOLDERS / typische structuur.
# Vervang ze met de exacte OrcaWave data names via F7 in de GUI.
# -------------------------------------------------

# Environment
diff.SetData("WaterDepth", 0, 30.0)

# Wave periods
# Vaak moet je eerst een count zetten en daarna de tabel vullen
diff.SetData("NumberOfPeriodsOrFrequencies", 0, len(wave_periods))
for i, T in enumerate(wave_periods):
    diff.SetData(f"PeriodOrFrequency", i, T)

# Wave headings
diff.SetData("NumberOfWaveHeadings", 0, len(wave_headings))
for i, hdg in enumerate(wave_headings):
    diff.SetData(f"WaveHeading", i, hdg)


diff.SetData("BodyInertiaSpecifiedBy", 0, "Matrix (for a general body)")
diff.SetData("BodyInertiaTensorOriginType", 0, "Centre of mass")
# Body inertia / mass properties
# Let op: body index / naam kan anders zijn in jouw model
diff.SetData("BodyCentreOfMassX", 0, com_x)
diff.SetData("BodyCentreOfMassY", 0, com_y)
diff.SetData("BodyCentreOfMassZ", 0, com_z)

diff.SetData("BodyMass", 0, mass)



diff.SetData("BodyInertiaTensorRx", 0, Ixx)   # xx
diff.SetData("BodyInertiaTensorRy", 1, Iyy)   # yy
diff.SetData("BodyInertiaTensorRz", 2, Izz)   # zz


print("Rx:", diff.BodyInertiaTensorRx[0])
print("Ry:", diff.BodyInertiaTensorRy[1])
print("Rz:", diff.BodyInertiaTensorRz[2])


# Optioneel: output settings
# Ook hier weer: exacte namen via F7
# set_value(diff, "CalculateDisplacementRAOs", "Yes")
# set_value(diff, "CalculateLoadRAOs", "Yes")
# set_value(diff, "CalculateAddedMassAndDamping", "Yes")
# set_value(diff, "CalculateWaveDriftQTFs", "No")

# Validation
errors = print_validation(diff)
if errors:
    raise RuntimeError("Model heeft validation errors. Fix eerst de data-itemnamen of invoer.")



Rx: 3289.0
Ry: 3334.0
Rz: 5409.0

=== VALIDATION INFO ===
('Estimated peak memory required during calculation: 854 MiB per thread.',)

=== VALIDATION WARNINGS ===
('Calculation mesh: the following panels have large aspect ratio, consider re-meshing: 737 759 781 803 825 847 869 891 913 935 957 979 1001 1023 1045 1067 1089 1111 1133 1155 1177 1199 1221 1243 1265 1287 1309 1331 1353 1375 1397 1419 2177 2199 2221 2243 2265 2287 2309 2331 2353 2375 2397 2419 2441 2463 2485 2507 2529 2551 2573 2595 2617 2639 2661 2683 2705 2727 2749 2771 2793 2815 2837 2859 3617 3639 3661 3683 3705 3727 3749 3771 3793 3815 3837 3859 3881 3903 3925 3947 3969 3991 4013 4035 4057 4079 4101 4123 4145 4167 4189 4211 4233 4255 4277 4299', 'Calculation mesh: the following panels are large compared to the wavelength of the shortest wave: 2 3 8-22 25 26 31-45 48 49 54-68 71 72 77-91 94 95 100-114 117 118 123-137 140 141 146-160 163 164 169-183 186 187 192-206 209 210 215-229 232 233 238-252 255 256 261-275 278 279 28

In [ ]:
# Run calculation
print("\n=== START CALCULATION ===")
diff.Calculate()
print("=== CALCULATION DONE ===")

# Save outputs
diff.SaveResults(str(owr_file))
diff.SaveResultsSpreadsheet(str(xlsx_file))

print("\nBestanden opgeslagen:")
print(f"  Results: {owr_file}")
print(f"  Spreadsheet: {xlsx_file}")


=== START CALCULATION ===
